# 01B. Natural Completion Suite (Continuous & Hard Cutoff)
Evaluates conditions: **continuous, cutoff** on exact 500 test questions (`max_new_tokens=800`).

In [1]:
file_suffix = 'nb1b'
!pip install -q evaluate bert_score bitsandbytes accelerate transformers rouge_score scipy
import os, sys, json, time, torch, numpy as np, pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import evaluate
print('PyTorch Version:', torch.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 50.3 MB/s eta 0:00:00
PyTorch Version: 2.10.0+cu128


In [2]:
possible_paths = [
    '/kaggle/input/datasets/trungkiennnn/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/datasets/anhemgithom/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/datasets/trungkiennnn/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese_medical_halueval_15k_specialized.json',
    'e:/Paper_Steering_VN_15K/data/vietnamese_medical_halueval_15k_specialized.json',
    './data/vietnamese_medical_halueval_15k_specialized.json',
    './vietnamese_medical_halueval_15k_specialized.json'
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    search_root = '/kaggle/input' if os.path.exists('/kaggle/input') else '.'
    for root, dirs, files in os.walk(search_root):
        for file in files:
            if file.endswith('.json') and ('halueval' in file.lower() or '15k' in file.lower() or 'medical' in file.lower() or 'vnese' in file.lower()):
                data_path = os.path.join(root, file)
                break
        if data_path: break

if data_path is None:
    raise FileNotFoundError("Dataset file not found! Please check Kaggle input data sidebar.")

print('✅ Resolved Dataset Path:', data_path)
with open(data_path, 'r', encoding='utf-8') as f: full_dataset = json.load(f)
test_data = full_dataset[-500:]
train_pool = full_dataset[:-2205]
print('Total dataset size:', len(full_dataset), '| Test size:', len(test_data))
assert len(test_data) == 500, 'Test size must be 500'


✅ Resolved Dataset Path: /kaggle/input/datasets/trungkiennnn/vnese-data/vietnamese_medical_halueval_15k_specialized.json
Total dataset size: 14700 | Test size: 500


In [3]:
model_id = 'Qwen/Qwen2.5-7B-Instruct'
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model.eval()
bertscore = evaluate.load('bertscore')
print('Model & BERTScore loaded!')


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model & BERTScore loaded!


In [4]:
pos_acts, neg_acts = [], []
for item in train_pool[:300]:
    q, pos_ans, neg_ans = item['question'], item.get('right_answer', item.get('positive_answer')), item['hallucinated_answer']
    t_pos = f'<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{pos_ans}'
    t_neg = f'<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{neg_ans}'
    with torch.no_grad():
        inp_p = tokenizer(t_pos, return_tensors='pt').to(model.device)
        pos_acts.append(model(inp_p.input_ids, output_hidden_states=True).hidden_states[8][0, -1, :].detach().cpu())
        inp_n = tokenizer(t_neg, return_tensors='pt').to(model.device)
        neg_acts.append(model(inp_n.input_ids, output_hidden_states=True).hidden_states[8][0, -1, :].detach().cpu())
v_diff = torch.stack(pos_acts).mean(dim=0) - torch.stack(neg_acts).mean(dim=0)
v_steer = v_diff / v_diff.norm(p=2)
print('Vector v_steer extracted!')


Vector v_steer extracted!


In [5]:
def make_safe_hook(schedule_type='decay', alpha_0=18.0, K=16):
    step = 0
    def hook(module, inp, out):
        nonlocal step; step += 1
        if schedule_type == 'continuous': alpha_t = alpha_0
        elif schedule_type == 'cutoff': alpha_t = alpha_0 if step <= K else 0.0
        elif schedule_type == 'decay': alpha_t = alpha_0 * (1.0 - (step - 1) / K) if 1 <= step <= K else 0.0
        elif schedule_type == 'matched_d1': alpha_t = 0.7650
        else: alpha_t = 0.0
        if alpha_t != 0.0:
            cur = out[0] if isinstance(out, tuple) else out
            v_curr = v_steer.to(device=cur.device, dtype=cur.dtype)
            mod = cur + alpha_t * v_curr
            return (mod,) + out[1:] if isinstance(out, tuple) else mod
        return out
    return hook


In [6]:
file_suffix = 'nb1b'
target_layer = model.model.layers[8]
conditions = ["continuous", "cutoff"]
all_records = []
def compute_rep4(text):
    toks = text.lower().split()
    if len(toks) < 4: return 0.0
    ngs = [tuple(toks[i:i+4]) for i in range(len(toks)-3)]
    return (1.0 - len(set(ngs)) / len(ngs)) * 100.0

for cond in conditions:
    print(f'🚀 Running 800-Token Evaluation for Schedule: {{cond}}...')
    for idx, item in enumerate(tqdm(test_data, desc=f'Evaluating {{cond}}')):
        prompt = f"<|im_start|>user\n{item['question']}<|im_end|>\n<|im_start|>assistant\n"
        inp = tokenizer(prompt, return_tensors='pt').to(model.device)
        p_len = inp.input_ids.shape[1]
        t0 = time.time()
        if cond != 'baseline':
            h_handle = target_layer.register_forward_hook(make_safe_hook(cond))
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=800, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        if cond != 'baseline': h_handle.remove()
        lat = time.time() - t0
        gen_toks = out[0][p_len:]
        gen_text = tokenizer.decode(gen_toks, skip_special_tokens=True)
        eos_hit = 1 if tokenizer.eos_token_id in gen_toks else 0
        all_records.append({
            'question_id': f'Q-{idx:03d}',
            'category': item.get('category', 'general'),
            'condition': cond,
            'generated_text': gen_text,
            'output_length': len(gen_toks),
            'eos_hit': eos_hit,
            'latency': lat,
            'gold_reference': item.get('right_answer', item.get('positive_answer')),
            'hallucinated_reference': item['hallucinated_answer'],
            'rep_4': compute_rep4(gen_text)
        })

os.makedirs('outputs', exist_ok=True)
out_name = f'outputs/natural_completion_{file_suffix}_outputs.jsonl'
with open(out_name, 'w', encoding='utf-8') as f:
    for r in all_records: f.write(json.dumps(r, ensure_ascii=False) + '\n')
print(f'Saved {out_name}!')


🚀 Running 800-Token Evaluation for Schedule: {cond}...


Evaluating {cond}: 100%|██████████| 500/500 [2:57:17<00:00, 21.28s/it]


🚀 Running 800-Token Evaluation for Schedule: {cond}...


Evaluating {cond}: 100%|██████████| 500/500 [3:26:02<00:00, 24.72s/it]

Saved outputs/natural_completion_nb1b_outputs.jsonl!


In [7]:
file_suffix = 'nb1b'
df = pd.DataFrame(all_records)
print('Computing BERTScore metrics...')
bs_p = bertscore.compute(predictions=df['generated_text'].tolist(), references=df['gold_reference'].tolist(), model_type='bert-base-multilingual-cased')['f1']
bs_n = bertscore.compute(predictions=df['generated_text'].tolist(), references=df['hallucinated_reference'].tolist(), model_type='bert-base-multilingual-cased')['f1']
df['BS_positive'] = bs_p
df['BS_negative'] = bs_n
df['RefPref'] = (df['BS_positive'] > df['BS_negative']).astype(int)
df['tie'] = (df['BS_positive'] == df['BS_negative']).astype(int)

summary = df.groupby('condition').agg(
    raw_ref_pref_count=('RefPref', 'sum'),
    ref_pref_pct=('RefPref', lambda x: x.mean()*100),
    bertscore_f1=('BS_positive', 'mean'),
    eos_hit_rate=('eos_hit', lambda x: x.mean()*100),
    rep_4=('rep_4', 'mean'),
    mean_output_length=('output_length', 'mean'),
    mean_latency=('latency', 'mean')
).reset_index()

os.makedirs('results', exist_ok=True)
sum_name = f'results/natural_{file_suffix}_summary.csv'
summary.to_csv(sum_name, index=False)
print(f'Saved {sum_name}!')
print(summary)
assert len(df['question_id'].unique()) == 500, 'Must have 500 unique questions'
print('✅ Split Notebook Assertions Passed 100%!')


Computing BERTScore metrics...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved results/natural_nb1b_summary.csv!
    condition  raw_ref_pref_count  ref_pref_pct  bertscore_f1  eos_hit_rate  \
0  continuous                 343          68.6      0.670496          99.8   
1      cutoff                 353          70.6      0.669492         100.0   

      rep_4  mean_output_length  mean_latency  
0  4.381163             278.566     21.273478  
1  5.349659             325.574     24.722250  
✅ Split Notebook Assertions Passed 100%!
